In [1]:
import pandas as pd
import numpy as np
import json
import re
from pathlib import Path
from collections import Counter

from sklearn.metrics import cohen_kappa_score
import krippendorff  # pip install krippendorff

from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import torch

pd.set_option("display.max_colwidth", 120)
print("✅ Imports OK")
print(f"CUDA available: {torch.cuda.is_available()}")

/opt/anaconda3/envs/tf312/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Imports OK
CUDA available: False


In [2]:
panel = pd.read_csv("humans_interviews.csv")
print(f"Panel shape: {panel.shape}")

COMP_COLS = [f"Q_comparaison_{i}" for i in range(1, 7)]

df = panel[["panelist_id"] + COMP_COLS].copy()
print(df.head(2))

Panel shape: (53, 24)


KeyError: "['panelist_id', 'Q_comparaison_2', 'Q_comparaison_6'] not in index"

In [ ]:

SCHEMA = {
    "creative_preference": {
        "values": ["Bouygues", "Orange", "Neutral", "Mixed"],
        "rule": (
            "Which brand does the panelist prefer from a CREATIVE/ENTERTAINMENT "
            "angle (humour, originality, storytelling, visual style)? "
            "'Mixed' = liked both equally for creative reasons. "
            "'Neutral' = no clear creative opinion expressed."
        )
    },
    "commercial_preference": {
        "values": ["Bouygues", "Orange", "Neutral", "Mixed"],
        "rule": (
            "Which brand does the panelist prefer from a COMMERCIAL/TRUST angle "
            "(reliability, purchase intent, brand image, fiabilité)? "
            "'Mixed' = balanced commercial appreciation. "
            "'Neutral' = no clear commercial opinion expressed."
        )
    },
    "overall_preference": {
        "values": ["Bouygues", "Orange", "Neutral", "Mixed"],
        "rule": (
            "What is the panelist's OVERALL preferred brand, taking everything "
            "into account? If they mention preferring one overall despite "
            "nuances, use that brand. 'Mixed' = genuinely undecided overall."
        )
    }
}

print("Schema defined:")
for dim, info in SCHEMA.items():
    print(f"  {dim}: {info['values']}")

Schema defined:
  creative_preference: ['Bouygues', 'Orange', 'Neutral', 'Mixed']
  commercial_preference: ['Bouygues', 'Orange', 'Neutral', 'Mixed']
  overall_preference: ['Bouygues', 'Orange', 'Neutral', 'Mixed']


In [ ]:
SYSTEM_PROMPT = """Tu es un annotateur expert en analyse de discours publicitaire.
Tu analyses des verbatims de panélistes ayant regardé deux publicités télévisées :
- Bouygues Telecom (humour absurde, scénario policier décalé)
- Orange (message de fiabilité réseau, ton rassurant)

Tu dois extraire 3 dimensions de préférence à partir du verbatim fourni.

SCHÉMA D'ANNOTATION :
{schema}

RÈGLES STRICTES :
1. Réponds UNIQUEMENT avec un objet JSON valide, sans texte avant ou après.
2. Chaque valeur doit être exactement l'une de : "Bouygues", "Orange", "Neutral", "Mixed".
3. Ajoute un champ "reasoning" (max 2 phrases) expliquant tes choix.
4. Ajoute un champ "confidence" (float 0.0–1.0) représentant ta certitude globale.

FORMAT DE SORTIE :
{{
  "creative_preference": "<valeur>",
  "commercial_preference": "<valeur>",
  "overall_preference": "<valeur>",
  "reasoning": "<explication courte>",
  "confidence": <float>
}}
""".format(
    schema="\n".join(
        f"- {dim}: {info['rule']}"
        for dim, info in SCHEMA.items()
    )
)

# Few-shot examples (3 labeled gold examples embedded in user prompt)
FEW_SHOT_EXAMPLES = [
    {
        "verbatim": "J'ai préféré celle de Bouygues. Elle était plus drôle, plus originale. "
                    "Mais honnêtement pour choisir un opérateur, Orange me rassure plus sur la fiabilité.",
        "label": {
            "creative_preference": "Bouygues",
            "commercial_preference": "Orange",
            "overall_preference": "Mixed",
            "reasoning": "Créativité clairement Bouygues, fiabilité commerciale Orange, pas de gagnant global.",
            "confidence": 0.95
        }
    },
    {
        "verbatim": "Bouygues, sans hésiter. Pour son concept, pour la fiabilité aussi — "
                    "je leur fais autant confiance qu'Orange pour le réseau. Et la pub est bien meilleure.",
        "label": {
            "creative_preference": "Bouygues",
            "commercial_preference": "Bouygues",
            "overall_preference": "Bouygues",
            "reasoning": "Préférence créative ET commerciale pour Bouygues, préférence globale sans ambiguïté.",
            "confidence": 0.92
        }
    },
    {
        "verbatim": "Je préfère celle d'Orange, sans hésiter. Elle parle de fiabilité réseau, "
                    "c'est concret. Bouygues c'était marrant mais ça ne me donne pas envie de changer d'opérateur.",
        "label": {
            "creative_preference": "Neutral",
            "commercial_preference": "Orange",
            "overall_preference": "Orange",
            "reasoning": "Bouygues amusant mais sans impact créatif fort. Orange préférée commercialement et globalement.",
            "confidence": 0.93
        }
    },
    {
        "verbatim": "Franchement les deux se valent. Bouygues m'a fait rire, Orange m'a rassuré. "
                    "Pour un abonnement je choisirais peut-être Bouygues parce que la marque m'a marqué.",
        "label": {
            "creative_preference": "Bouygues",
            "commercial_preference": "Mixed",
            "overall_preference": "Bouygues",
            "reasoning": "Créativité Bouygues, commercial équilibré donc Mixed, mais impact mémoriel penche vers Bouygues globalement.",
            "confidence": 0.85
        }
    }
]

def build_user_prompt(verbatim: str, include_few_shot: bool = True) -> str:
    prompt = ""
    if include_few_shot:
        prompt += "### EXEMPLES ANNOTÉS\n\n"
        for ex in FEW_SHOT_EXAMPLES:
            prompt += f"Verbatim: {ex['verbatim']}\n"
            prompt += f"Annotation: {json.dumps(ex['label'], ensure_ascii=False)}\n\n"
        prompt += "---\n\n"
    prompt += f"### VERBATIM À ANNOTER\n\n{verbatim}\n\nAnnotation:"
    return prompt

print("✅ System prompt and few-shot examples built.")
print(f"\nSystem prompt preview:\n{SYSTEM_PROMPT[:400]}...")

✅ System prompt and few-shot examples built.

System prompt preview:
Tu es un annotateur expert en analyse de discours publicitaire.
Tu analyses des verbatims de panélistes ayant regardé deux publicités télévisées :
- Bouygues Telecom (humour absurde, scénario policier décalé)
- Orange (message de fiabilité réseau, ton rassurant)

Tu dois extraire 3 dimensions de préférence à partir du verbatim fourni.

SCHÉMA D'ANNOTATION :
- creative_preference: Which brand does ...


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.2"

device = torch.device("mps") if torch.backends.mps.is_available() else torch.device("cpu")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,  # MPS supports float16
).to(device)
model.eval()
print(f"✅ Model on: {device}")

Loading weights: 100%|██████████| 291/291 [00:25<00:00, 11.29it/s]


✅ Model on: mps


In [ ]:
# Run if mem error above
# del model
# del tokenizer
# import torch, gc
# gc.collect()
# torch.mps.empty_cache()
# print("✅ MPS memory freed")

✅ MPS memory freed


In [ ]:
import requests, time

def annotate_verbatim(verbatim: str) -> dict:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": build_user_prompt(verbatim)}
    ]
    resp = requests.post(
        "http://localhost:11434/api/chat",
        json={"model": "mistral", "messages": messages,
              "stream": False, "options": {"temperature": 0.1}},
        timeout=120
    )
    generated = resp.json()["message"]["content"]

    parsed = extract_json(generated)
    if parsed is None:
        return {"creative_preference": "ParseError",
                "commercial_preference": "ParseError",
                "overall_preference":    "ParseError",
                "reasoning":             generated[:200],
                "confidence":            0.0,
                "raw_output":            generated}

    parsed = validate_labels(parsed)
    confidence = float(parsed.get("confidence", 0.0))
    for dim in SCHEMA:
        parsed[f"{dim}_flagged"] = confidence < CONFIDENCE_THRESHOLD
    parsed["raw_output"] = generated
    return parsed

# Smoke test — should finish in ~5–10s
t0 = time.time()
test = annotate_verbatim("J'ai préféré Bouygues pour l'humour, mais Orange pour la fiabilité.")
print(f"⏱ {time.time()-t0:.1f}s")
print(json.dumps({k: v for k, v in test.items() if k != "raw_output"}, ensure_ascii=False, indent=2))

⏱ 14.8s
{
  "creative_preference": "Bouygues",
  "commercial_preference": "Orange",
  "overall_preference": "Mixed",
  "reasoning": "Créativité clairement Bouygues, fiabilité commerciale Orange, pas de gagnant global.",
  "confidence": 0.95,
  "creative_preference_flagged": false,
  "commercial_preference_flagged": false,
  "overall_preference_flagged": false
}


In [ ]:
COMP_COLS = [f"Q_comparaison_{i}" for i in range(1, 7)]

def build_verbatim(row):
    parts = [str(row[col]) for col in COMP_COLS if pd.notna(row[col])]
    return " ".join(parts)

df["verbatim"] = df.apply(build_verbatim, axis=1)
print(f"✅ Verbatims built. Sample:\n{df['verbatim'].iloc[0][:300]}...")

✅ Verbatims built. Sample:
J'ai préféré celle de Bouygues. Elle était plus marrante, plus originale. Je l'ai regardée comme un petit sketch, c'était plus sympa. L'autre, celle d'Orange, est efficace mais un peu... banale dans la forme. Celle de Bouygues, sans hésiter. L'idée de la scène de crime pour du wifi, c'est tellement ...


In [ ]:
PILOT_N = 30
pilot_df = df.sample(n=PILOT_N, random_state=42).copy().reset_index(drop=True)

results = []
for i, row in pilot_df.iterrows():
    t0 = time.time()
    result = annotate_verbatim(row["verbatim"])
    result["panelist_id"] = row["panelist_id"]
    results.append(result)
    elapsed = time.time() - t0
    print(f"[{i+1:02d}/{PILOT_N}] {elapsed:.1f}s — "
          f"creative={result['creative_preference']} | "
          f"commercial={result['commercial_preference']} | "
          f"overall={result['overall_preference']} | "
          f"conf={result.get('confidence','?')}")

pilot_results = pd.DataFrame(results)
pilot_results.to_csv("pilot_annotations_30.csv", index=False)
print(f"\n✅ Pilot done. Shape: {pilot_results.shape}")
pilot_results[["panelist_id","creative_preference",
               "commercial_preference","overall_preference","confidence"]].head(10)

[01/30] 6.4s — creative=Orange | commercial=Orange | overall=Orange | conf=0.98
[02/30] 5.7s — creative=Orange | commercial=Orange | overall=Orange | conf=0.98
[03/30] 5.5s — creative=Bouygues | commercial=Orange | overall=Orange | conf=0.98
[04/30] 5.2s — creative=Orange | commercial=Orange | overall=Orange | conf=0.98
[05/30] 5.9s — creative=Orange | commercial=Orange | overall=Orange | conf=0.98
[06/30] 5.9s — creative=Bouygues | commercial=Orange | overall=Orange | conf=0.95
[07/30] 5.6s — creative=Bouygues | commercial=Orange | overall=Orange | conf=0.95
[08/30] 5.8s — creative=Bouygues | commercial=Orange | overall=Orange | conf=0.95
[09/30] 5.6s — creative=Orange | commercial=Orange | overall=Orange | conf=0.98
[10/30] 6.1s — creative=Bouygues | commercial=Orange | overall=Orange | conf=0.95
[11/30] 6.1s — creative=Bouygues | commercial=Neutral | overall=Bouygues | conf=0.98
[12/30] 6.2s — creative=Bouygues | commercial=Orange | overall=Orange | conf=0.98
[13/30] 5.9s — creative

,panelist_id,creative_preference,commercial_preference,overall_preference,confidence
0,67b0d131d362f5886c2e5c61,Orange,Orange,Orange,0.98
1,67b0d12fd362f5886c2e5bdd,Orange,Orange,Orange,0.98
2,67b0d125d362f5886c2e59f3,Bouygues,Orange,Orange,0.98
3,67b0d12dd362f5886c2e5b91,Orange,Orange,Orange,0.98
4,67b0d125d362f5886c2e59ff,Orange,Orange,Orange,0.98
5,67b0d133d362f5886c2e5c9c,Bouygues,Orange,Orange,0.95
6,67b0d13ad362f5886c2e5dd3,Bouygues,Orange,Orange,0.95
7,67b0d131d362f5886c2e5c41,Bouygues,Orange,Orange,0.95
8,67b0d136d362f5886c2e5d2b,Orange,Orange,Orange,0.98
9,67b0d132d362f5886c2e5c88,Bouygues,Orange,Orange,0.95


In [ ]:
# Cohen's κ — using synthetic labels for now (replace after manual annotation)
import random
random.seed(42)
LABEL_VALS = ["Bouygues", "Orange", "Neutral", "Mixed"]
human_demo = pilot_results[["panelist_id"]].copy()
for dim in SCHEMA:
    human_demo[f"{dim}_human"] = [random.choice(LABEL_VALS) for _ in range(PILOT_N)]
merged_pilot = pilot_results.merge(human_demo, on="panelist_id")

# Export verbatims for your manual annotation
export = df[df["panelist_id"].isin(pilot_results["panelist_id"])][
    ["panelist_id", "verbatim"]
].copy()
export["creative_preference"]   = ""
export["commercial_preference"] = ""
export["overall_preference"]    = ""
export.to_csv("pilot_to_annotate_manually.csv", index=False)
print("✅ Exported to pilot_to_annotate_manually.csv — fill in the labels then re-run this cell")

✅ Exported to pilot_to_annotate_manually.csv — fill in the labels then re-run this cell


In [ ]:
def interpret_kappa(k):
    if k >= 0.80: return "Near-perfect ✅"
    if k >= 0.61: return "Substantial ✅"
    if k >= 0.41: return "Moderate ⚠️  — acceptable"
    if k >= 0.21: return "Fair ❌ — revise schema"
    return "Slight/Poor ❌ — do NOT proceed"

print("=" * 65)
print(f"{'Dimension':<30} {'κ':>6}  Interpretation")
print("=" * 65)
schema_ok = True
for dim, k in kappa_results.items():
    interp = interpret_kappa(k)
    print(f"{dim:<30} {k:>6.3f}  {interp}")
    if k < 0.41:
        schema_ok = False
print("=" * 65)
print("✅ Schema valid — proceed to full run." if schema_ok
      else "❌ Revise dimension guidelines before full run.")

Dimension                           κ  Interpretation
creative_preference             0.071  Slight/Poor ❌ — do NOT proceed
commercial_preference           0.027  Slight/Poor ❌ — do NOT proceed
overall_preference              0.016  Slight/Poor ❌ — do NOT proceed
❌ Revise dimension guidelines before full run.


In [ ]:
COMP_COLS = [f"Q_comparaison_{i}" for i in range(1, 7)]

def build_verbatim(row):
    parts = [str(row[col]) for col in COMP_COLS if pd.notna(row[col])]
    return " ".join(parts)

df["verbatim"] = df.apply(build_verbatim, axis=1)
print(f"✅ Verbatims rebuilt. Sample:\n{df['verbatim'].iloc[0][:300]}...")

✅ Verbatims rebuilt. Sample:
J'ai préféré celle de Bouygues. Elle était plus marrante, plus originale. Je l'ai regardée comme un petit sketch, c'était plus sympa. L'autre, celle d'Orange, est efficace mais un peu... banale dans la forme. Celle de Bouygues, sans hésiter. L'idée de la scène de crime pour du wifi, c'est tellement ...


In [ ]:
pilot_with_verbatim = merged_pilot.merge(
    df[["panelist_id", "verbatim"]], on="panelist_id", how="left"
)

for dim in SCHEMA:
    disagree = pilot_with_verbatim[
        pilot_with_verbatim[dim] != pilot_with_verbatim[f"{dim}_human"]
    ]
    if len(disagree):
        print(f"\n── {dim}: {len(disagree)} disagreements ──")
        for _, row in disagree.head(3).iterrows():
            print(f"  LLM={row[dim]}, Human={row[f'{dim}_human']}")
            print(f"  Verbatim: {str(row.get('verbatim',''))[:180]}...")
            print(f"  Reasoning: {row.get('reasoning','')}\n")


── creative_preference: 18 disagreements ──
  LLM=Orange, Human=Bouygues
  Verbatim: Celle d'Orange, sans aucune hésitation. Elle est plus simple, plus vraie. Elle parle de la vie des gens. L'autre, c'est de l'esbroufe. Celle d'Orange aussi, je pense. Parce qu'on r...
  Reasoning: Créativité plus convaincante pour Orange, message commercial clairement centré sur fiabilité et service.

  LLM=Orange, Human=Bouygues
  Verbatim: J'ai préféré la publicité d'Orange, sans hésiter. Elle est plus simple, plus claire, et surtout plus humaine. Je me suis reconnue dans les situations. L'autre, celle de Bouygues, é...
  Reasoning: Simplicité et humanité de la publicité Orange, plus convaincante pour le spectateur. Bouygues est original mais trop compliquée et négative.

  LLM=Bouygues, Human=Neutral
  Verbatim: Je préfère celle d'Orange. Même si celle de Bouygues est plus originale, celle d'Orange me parle plus directement. Elle touche à des points sensibles de mon quotidien, notamment l'...
  Rea

In [ ]:
from collections import Counter

print("PILOT DISTRIBUTION (30 samples)")
print("=" * 45)
for dim in SCHEMA:
    counts = Counter(pilot_results[dim])
    print(f"\n{dim}:")
    for label, n in sorted(counts.items(), key=lambda x: -x[1]):
        bar = "█" * n
        print(f"  {label:<12} {n:>2}  {bar}")

PILOT DISTRIBUTION (30 samples)

creative_preference:
  Bouygues     19  ███████████████████
  Orange       11  ███████████

commercial_preference:
  Orange       26  ██████████████████████████
  Mixed         2  ██
  Neutral       1  █
  Bouygues      1  █

overall_preference:
  Orange       25  █████████████████████████
  Bouygues      5  █████


In [ ]:
all_results = []
CHECKPOINT_EVERY = 50
checkpoint_path  = "annotations_checkpoint.csv"

total     = len(df)
run_start = time.time()

for i, row in df.iterrows():
    t0     = time.time()
    result = annotate_verbatim(row["verbatim"])
    result["panelist_id"] = row["panelist_id"]
    all_results.append(result)
    elapsed = time.time() - t0

    if (i + 1) % CHECKPOINT_EVERY == 0:
        pd.DataFrame(all_results).to_csv(checkpoint_path, index=False)
        done = i + 1
        avg  = (time.time() - run_start) / done
        eta  = avg * (total - done) / 60
        print(f"[{done:>3}/{total}] checkpoint saved | "
              f"avg {avg:.1f}s/sample | ETA ~{eta:.0f} min")

annotations = pd.DataFrame(all_results)
annotations.to_csv("preference_annotations_multidim.csv", index=False)
print(f"\n✅ Full annotation done. Shape: {annotations.shape}")

[ 50/800] checkpoint saved | avg 6.0s/sample | ETA ~75 min
[100/800] checkpoint saved | avg 5.9s/sample | ETA ~68 min
[150/800] checkpoint saved | avg 5.8s/sample | ETA ~63 min
[200/800] checkpoint saved | avg 7.1s/sample | ETA ~71 min
[250/800] checkpoint saved | avg 10.2s/sample | ETA ~94 min
[300/800] checkpoint saved | avg 11.5s/sample | ETA ~96 min
[350/800] checkpoint saved | avg 11.2s/sample | ETA ~84 min
[400/800] checkpoint saved | avg 10.5s/sample | ETA ~70 min
[450/800] checkpoint saved | avg 10.1s/sample | ETA ~59 min
[500/800] checkpoint saved | avg 9.7s/sample | ETA ~49 min
[550/800] checkpoint saved | avg 9.4s/sample | ETA ~39 min
[600/800] checkpoint saved | avg 9.1s/sample | ETA ~30 min
[650/800] checkpoint saved | avg 8.9s/sample | ETA ~22 min
[700/800] checkpoint saved | avg 8.7s/sample | ETA ~14 min
[750/800] checkpoint saved | avg 8.5s/sample | ETA ~7 min
[800/800] checkpoint saved | avg 10.6s/sample | ETA ~0 min

✅ Full annotation done. Shape: (800, 10)


In [ ]:
final = pd.read_csv("preference_annotations_multidim.csv")
total = len(final)

print("=" * 55)
print("ANNOTATION QUALITY REPORT")
print("=" * 55)
for dim in SCHEMA:
    errors      = (final[dim] == "ParseError").sum()
    uncertain   = (final[dim] == "Uncertain").sum()
    flagged_col = f"{dim}_flagged"
    low_conf    = final[flagged_col].sum() if flagged_col in final.columns else "N/A"
    dist        = final[dim].value_counts(normalize=True).round(3).to_dict()
    print(f"\n[{dim}]")
    print(f"  ParseErrors    : {errors} ({errors/total:.1%})")
    print(f"  Uncertain      : {uncertain} ({uncertain/total:.1%})")
    print(f"  Low confidence : {low_conf}")
    print(f"  Distribution   : {dist}")

print(f"\nMean confidence      : {final['confidence'].mean():.3f}")
print(f"Low conf (<{CONFIDENCE_THRESHOLD}) : {(final['confidence'] < CONFIDENCE_THRESHOLD).sum()}")

ANNOTATION QUALITY REPORT

[creative_preference]
  ParseErrors    : 0 (0.0%)
  Uncertain      : 0 (0.0%)
  Low confidence : 0
  Distribution   : {'Bouygues': 0.6, 'Orange': 0.34, 'Mixed': 0.06}

[commercial_preference]
  ParseErrors    : 0 (0.0%)
  Uncertain      : 0 (0.0%)
  Low confidence : 0
  Distribution   : {'Orange': 0.838, 'Mixed': 0.109, 'Bouygues': 0.046, 'Neutral': 0.008}

[overall_preference]
  ParseErrors    : 0 (0.0%)
  Uncertain      : 0 (0.0%)
  Low confidence : 0
  Distribution   : {'Orange': 0.71, 'Bouygues': 0.194, 'Mixed': 0.096}

Mean confidence      : 0.966
Low conf (<0.7) : 0


In [ ]:
panel_merged = panel.merge(
    final[["panelist_id", "creative_preference", "commercial_preference",
           "overall_preference", "confidence", "reasoning"]],
    on="panelist_id",
    how="left"
)

panel_merged.to_csv("interviews_with_multidim_preferences.csv", index=False)
print(f"✅ Final dataset saved.")
print(f"   Shape      : {panel_merged.shape}")
print(f"   Merge rate : {panel_merged['overall_preference'].notna().mean():.1%}")
panel_merged[["panelist_id","creative_preference",
              "commercial_preference","overall_preference","confidence"]].head(5)

✅ Final dataset saved.
   Shape      : (800, 53)
   Merge rate : 100.0%


,panelist_id,creative_preference,commercial_preference,overall_preference,confidence
0,67b0d135d362f5886c2e5cfe,Bouygues,Orange,Mixed,0.98
1,67b0d12ed362f5886c2e5b9c,Bouygues,Orange,Mixed,0.95
2,67b0d12bd362f5886c2e5b09,Bouygues,Orange,Orange,0.98
3,67b0d127d362f5886c2e5a4a,Orange,Orange,Orange,0.98
4,67b0d12cd362f5886c2e5b4a,Bouygues,Orange,Mixed,0.95


In [ ]:
ct = pd.crosstab(
    panel_merged["creative_preference"],
    panel_merged["overall_preference"],
    margins=True,
    normalize="index"
).round(3)

print("Creative preference → Overall preference (row-normalized)")
print(ct)

Creative preference → Overall preference (row-normalized)
overall_preference   Bouygues  Mixed  Orange
creative_preference                         
Bouygues                0.321  0.160   0.519
Mixed                   0.021  0.000   0.979
Orange                  0.000  0.000   1.000
All                     0.194  0.096   0.710


In [ ]:
demo_cols = ["age", "gender", "csp", "education", "incomelevel", "location.citysize"]

for col in demo_cols:
    if col not in panel_merged.columns:
        continue
    print(f"\n── overall_preference by {col} ──")
    ct = pd.crosstab(panel_merged[col], panel_merged["overall_preference"],
                     normalize="index").round(3)
    print(ct)


── overall_preference by age ──
overall_preference  Bouygues  Mixed  Orange
age                                        
18                     0.500  0.000   0.500
19                     0.476  0.190   0.333
21                     0.667  0.333   0.000
22                     0.439  0.195   0.366
23                     0.375  0.375   0.250
..                       ...    ...     ...
77                     0.000  0.000   1.000
78                     0.000  0.000   1.000
80                     0.000  0.000   1.000
82                     0.000  0.000   1.000
83                     0.000  0.000   1.000

[61 rows x 3 columns]

── overall_preference by gender ──
overall_preference  Bouygues  Mixed  Orange
gender                                     
Female                 0.219  0.126   0.655
Male                   0.145  0.054   0.802
Non-binary             0.410  0.246   0.344

── overall_preference by csp ──
overall_preference                                           Bouygues  Mixed  \
csp

In [ ]:
# Panelists whose creative ≠ overall preference (the "split" cases)
panel_merged["creative_commercial_split"] = (
    panel_merged["creative_preference"] != panel_merged["overall_preference"]
).astype(int)

split_rate = panel_merged["creative_commercial_split"].mean()
print(f"Panelists with creative ≠ overall preference: {split_rate:.1%}")

# What do split panelists look like demographically?
if "age" in panel_merged.columns:
    print(f"\nMean age — split: {panel_merged[panel_merged['creative_commercial_split']==1]['age'].mean():.1f}")
    print(f"Mean age — aligned: {panel_merged[panel_merged['creative_commercial_split']==0]['age'].mean():.1f}")

Panelists with creative ≠ overall preference: 46.8%

Mean age — split: 43.0
Mean age — aligned: 52.1


In [ ]:
summary = pd.DataFrame({
    "Dimension": list(SCHEMA.keys()),
    "Bouygues %": [
        (final["creative_preference"]   == "Bouygues").mean(),
        (final["commercial_preference"] == "Bouygues").mean(),
        (final["overall_preference"]    == "Bouygues").mean(),
    ],
    "Orange %": [
        (final["creative_preference"]   == "Orange").mean(),
        (final["commercial_preference"] == "Orange").mean(),
        (final["overall_preference"]    == "Orange").mean(),
    ],
    "Mixed %": [
        (final["creative_preference"]   == "Mixed").mean(),
        (final["commercial_preference"] == "Mixed").mean(),
        (final["overall_preference"]    == "Mixed").mean(),
    ],
    "Neutral %": [
        (final["creative_preference"]   == "Neutral").mean(),
        (final["commercial_preference"] == "Neutral").mean(),
        (final["overall_preference"]    == "Neutral").mean(),
    ],
}).set_index("Dimension").round(3) * 100

print(summary.to_string())
summary.to_csv("preference_summary_table.csv")
print("\n✅ Saved to preference_summary_table.csv")

                       Bouygues %  Orange %  Mixed %  Neutral %
Dimension                                                      
creative_preference          60.0      34.0      6.0        0.0
commercial_preference         4.6      83.8     10.9        0.8
overall_preference           19.4      71.0      9.6        0.0

✅ Saved to preference_summary_table.csv


In [ ]:
import pandas as pd
import plotly.io as pio
import plotly.graph_objects as go
import plotly.express as px
import json

final = pd.read_csv("preference_annotations_multidim.csv")
panel = pd.read_csv("interviews_with_multidim_preferences.csv")

pio.templates.default = "plotly_white"
colors    = ["#0055A4", "#FF6600", "#888888", "#CCCCCC"]
color_map = {"Bouygues": colors[0], "Orange": colors[1], "Mixed": colors[2], "Neutral": colors[3]}

LEGEND_BELOW = dict(
    orientation='h',
    yanchor='top', y=-0.15,
    xanchor='center', x=0.5
)
MARGINS = dict(t=80, b=120, l=60, r=20)

# ── 1. Stacked distribution ───────────────────────────────────────────────────
dims         = ["creative_preference", "commercial_preference", "overall_preference"]
dim_labels   = ["Creative", "Commercial", "Overall"]
labels_order = ["Bouygues", "Orange", "Mixed", "Neutral"]

fig1 = go.Figure()
for label in labels_order:
    vals = [(final[d] == label).mean() * 100 for d in dims]
    fig1.add_trace(go.Bar(name=label, x=dim_labels, y=vals, marker_color=color_map[label]))
fig1.update_layout(
    barmode="stack",
    title=dict(text="Preference by Dimension (n=800)", x=0, xanchor="left"),
    legend=LEGEND_BELOW,
    margin=MARGINS
)
fig1.update_yaxes(title_text="Panelists (%)")
fig1.update_xaxes(title_text="Dimension")
fig1.write_image("fig_stacked_distribution.png")
with open("fig_stacked_distribution.png.meta.json", "w") as f:
    json.dump({"caption": "Stacked preference distribution across three annotation dimensions (n=800)",
               "description": "Stacked bar showing Bouygues, Orange, Mixed, Neutral % for creative, commercial, overall."}, f)

# ── 2. Creative → Overall heatmap ─────────────────────────────────────────────
ct     = pd.crosstab(final["creative_preference"], final["overall_preference"])
ct     = ct.reindex(index=["Bouygues","Orange","Mixed"], columns=["Bouygues","Orange","Mixed"], fill_value=0)
ct_pct = ct.div(ct.sum(axis=1), axis=0).mul(100).round(1)
text_vals = [[f"{v:.0f}%" for v in row] for row in ct_pct.values]

fig2 = go.Figure(go.Heatmap(
    z=ct_pct.values,
    x=ct_pct.columns.tolist(),
    y=ct_pct.index.tolist(),
    text=text_vals,
    texttemplate="%{text}",
    textfont=dict(size=14),
    colorscale="Blues",
    showscale=False
))
fig2.update_layout(
    title=dict(text="Creative → Overall Preference (row %)", x=0, xanchor="left"),
    margin=dict(t=80, b=60, l=100, r=20)
)
fig2.update_xaxes(title_text="Overall preference")
fig2.update_yaxes(title_text="Creative preference")
fig2.write_image("fig_crosstab_heatmap.png")
with open("fig_crosstab_heatmap.png.meta.json", "w") as f:
    json.dump({"caption": "Heatmap: creative vs overall preference (row-normalised %)",
               "description": "Row-normalised heatmap showing transition from creative to overall preference labels."}, f)

# ── 3. Overall preference by age group ───────────────────────────────────────
panel["age_group"] = pd.cut(panel["age"], bins=[17,29,39,49,65],
                             labels=["18-29","30-39","40-49","50-65"])
age_overall = (panel.groupby("age_group", observed=False)["overall_preference"]
               .value_counts(normalize=True).mul(100).reset_index())
age_overall.columns = ["age_group", "overall_preference", "pct"]
age_overall = age_overall[age_overall["overall_preference"].isin(["Bouygues","Orange","Mixed"])]

fig3 = go.Figure()
for label in ["Bouygues", "Orange", "Mixed"]:
    sub = age_overall[age_overall["overall_preference"] == label]
    fig3.add_trace(go.Bar(
        name=label,
        x=sub["age_group"].astype(str),
        y=sub["pct"],
        marker_color=color_map[label]
    ))
fig3.update_layout(
    barmode="group",
    title=dict(text="Overall Preference by Age Group", x=0, xanchor="left"),
    legend=LEGEND_BELOW,
    margin=MARGINS
)
fig3.update_yaxes(title_text="Panelists (%)")
fig3.update_xaxes(title_text="Age group")
fig3.write_image("fig_age_overall.png")
with open("fig_age_overall.png.meta.json", "w") as f:
    json.dump({"caption": "Overall preference distribution by age group",
               "description": "Grouped bar chart of overall brand preference by age bands."}, f)

# ── 4. Creative/overall split rate by gender ──────────────────────────────────
panel["creative_overall_split"] = (
    panel["creative_preference"] != panel["overall_preference"]
).astype(int)
gender_split = panel.groupby("gender")["creative_overall_split"].mean().reset_index()
gender_split.columns = ["gender", "split_rate"]
gender_split["split_rate"] *= 100

fig4 = go.Figure()
for i, row in gender_split.iterrows():
    fig4.add_trace(go.Bar(
        name=row["gender"], x=[row["gender"]], y=[row["split_rate"]],
        marker_color=colors[i % len(colors)], showlegend=False
    ))
fig4.update_layout(
    title=dict(text="Creative≠Overall Split Rate by Gender", x=0, xanchor="left"),
    margin=dict(t=80, b=60, l=60, r=20)
)
fig4.update_yaxes(title_text="Split rate (%)")
fig4.update_xaxes(title_text="Gender")
fig4.write_image("fig_split_gender.png")
with open("fig_split_gender.png.meta.json", "w") as f:
    json.dump({"caption": "Creative vs overall preference split rate by gender",
               "description": "Bar chart showing % of panelists per gender with differing creative and overall preferences."}, f)

# ── 5. Confidence distribution ────────────────────────────────────────────────
fig5 = go.Figure(go.Histogram(x=final["confidence"], nbinsx=20, marker_color=colors[0]))
fig5.update_layout(
    title=dict(text="Annotation Confidence Distribution (n=800)", x=0, xanchor="left"),
    margin=dict(t=80, b=60, l=60, r=20)
)
fig5.update_yaxes(title_text="Count")
fig5.update_xaxes(title_text="Confidence")
fig5.write_image("fig_confidence.png")
with open("fig_confidence.png.meta.json", "w") as f:
    json.dump({"caption": "Distribution of LLM annotation confidence scores across 800 panelists",
               "description": "Histogram of confidence values from the Mistral-7B annotation run."}, f)

# ── 6. Creative preference by CSP ─────────────────────────────────────────────
csp_abbrev = {
    "Cadres et professions intellectuelles supérieures": "Cadres",
    "Professions intermédiaires": "Prof. interm.",
    "Employés": "Employés",
    "Ouvriers": "Ouvriers",
    "Artisans, commerçants et chefs d'entreprise": "Artisans/Com.",
    "Agriculteurs exploitants": "Agriculteurs",
    "Retraités": "Retraités",
    "Sans activité professionnelle": "Sans activité",
}
panel["csp_short"] = panel["csp"].map(csp_abbrev).fillna(panel["csp"].str[:18])

csp_creative = (panel.groupby("csp_short")["creative_preference"]
                .value_counts(normalize=True).mul(100).reset_index())
csp_creative.columns = ["csp_short", "creative_preference", "pct"]
csp_creative = csp_creative[csp_creative["creative_preference"].isin(["Bouygues","Orange"])]

fig6 = go.Figure()
for label in ["Bouygues", "Orange"]:
    sub = csp_creative[csp_creative["creative_preference"] == label]
    fig6.add_trace(go.Bar(
        name=label,
        y=sub["csp_short"],
        x=sub["pct"],
        orientation='h',
        marker_color=color_map[label]
    ))
fig6.update_layout(
    barmode="group",
    title=dict(text="Creative Preference by CSP", x=0, xanchor="left"),
    legend=LEGEND_BELOW,
    margin=dict(t=80, b=100, l=130, r=20)
)
fig6.update_xaxes(title_text="Panelists (%)")
fig6.update_yaxes(title_text="")
fig6.write_image("fig_csp_creative.png")
with open("fig_csp_creative.png.meta.json", "w") as f:
    json.dump({"caption": "Creative preference by socio-professional category (CSP)",
               "description": "Horizontal grouped bar chart comparing Bouygues vs Orange creative preference across CSP categories."}, f)

print("✅ All 6 charts saved.")

✅ All 6 charts saved.


In [ ]:
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
import json

final = pd.read_csv("preference_annotations_multidim.csv")
panel = pd.read_csv("interviews_with_multidim_preferences.csv")

pio.templates.default = "plotly_white"
colors    = ["#0055A4", "#FF6600", "#888888", "#CCCCCC"]
color_map = {"Bouygues": colors[0], "Orange": colors[1], "Mixed": colors[2], "Neutral": colors[3]}
LEGEND_BELOW = dict(orientation='h', yanchor='top', y=-0.18, xanchor='center', x=0.5)

# ── 7. Donut: Overall preference share ───────────────────────────────────────
ov_counts = final["overall_preference"].value_counts()
fig7 = go.Figure(go.Pie(
    labels=ov_counts.index.tolist(),
    values=ov_counts.values.tolist(),
    hole=0.45,
    marker_colors=[color_map.get(l, "#CCCCCC") for l in ov_counts.index]
))
fig7.update_layout(
    title=dict(text="Overall Preference Share (n=800)", x=0, xanchor="left"),
    legend=LEGEND_BELOW,
    margin=dict(t=80, b=100, l=20, r=20)
)
fig7.update_traces(textinfo="percent+label", textfont_size=13)
fig7.write_image("fig_donut_overall.png")
with open("fig_donut_overall.png.meta.json", "w") as f:
    json.dump({"caption": "Donut chart of overall preference share across 800 panelists",
               "description": "Proportional split of Bouygues, Orange, Mixed overall preference."}, f)

# ── 8. Commercial preference by urban/rural classification ───────────────────
if "location.urban_rural_classification" in panel.columns:
    urb = (panel.groupby("location.urban_rural_classification")["commercial_preference"]
           .value_counts(normalize=True).mul(100).reset_index())
    urb.columns = ["urban_rural", "commercial_preference", "pct"]
    urb = urb[urb["commercial_preference"].isin(["Bouygues","Orange","Mixed"])]

    fig8 = go.Figure()
    for label in ["Bouygues","Orange","Mixed"]:
        sub = urb[urb["commercial_preference"] == label]
        fig8.add_trace(go.Bar(
            name=label, x=sub["urban_rural"], y=sub["pct"],
            marker_color=color_map[label]
        ))
    fig8.update_layout(
        barmode="group",
        title=dict(text="Commercial Preference by Urban/Rural", x=0, xanchor="left"),
        legend=LEGEND_BELOW,
        margin=dict(t=80, b=120, l=60, r=20)
    )
    fig8.update_yaxes(title_text="Panelists (%)")
    fig8.update_xaxes(title_text="Classification")
    fig8.write_image("fig_urban_commercial.png")
    with open("fig_urban_commercial.png.meta.json", "w") as f:
        json.dump({"caption": "Commercial preference by urban/rural classification",
                   "description": "Grouped bar chart of commercial brand preference by urban vs rural respondents."}, f)

# ── 9. Scatter: confidence vs creative=Bouygues rate per income level ─────────
if "income_level" in panel.columns:
    grp = panel.groupby("income_level").agg(
        mean_conf=("confidence", "mean"),
        bouygues_creative=("creative_preference", lambda x: (x=="Bouygues").mean()*100),
        n=("panelist_id", "count")
    ).reset_index()

    fig9 = go.Figure()
    fig9.add_trace(go.Scatter(
        x=grp["mean_conf"],
        y=grp["bouygues_creative"],
        mode="markers+text",
        text=grp["income_level"],
        textposition="top center",
        marker=dict(size=grp["n"]/4, color=colors[0], opacity=0.7),
    ))
    fig9.update_layout(
        title=dict(text="Confidence vs Bouygues Creative Rate by Income", x=0, xanchor="left"),
        margin=dict(t=80, b=80, l=70, r=20)
    )
    fig9.update_xaxes(title_text="Mean confidence")
    fig9.update_yaxes(title_text="Bouygues creative (%)")
    fig9.write_image("fig_scatter_income.png")
    with open("fig_scatter_income.png.meta.json", "w") as f:
        json.dump({"caption": "Mean annotation confidence vs Bouygues creative preference rate by income level",
                   "description": "Bubble scatter where size = group count, x = mean confidence, y = % choosing Bouygues creatively."}, f)

print("✅ 3 extra charts saved.")

✅ 3 extra charts saved.
